# Ndërtimi i Dataset-it Final të Balancuar

Bazuar në analizën e `explore.ipynb`, ndërtojmë një dataset single-label, të balancuar, me 9 kategori tematike universale (të verifikuara nëpër shumë portale).

## 1. Mapping-u i kategorive

Harta që bashkon variantet/sinonimet e kategorive (nga portale të ndryshme) në 9 kategoritë finale.

In [2]:
import pandas as pd

KATEGORI_MAPPING = {
    # Sport
    'Sport': 'Sport', 'Sporti': 'Sport', 'Premier League': 'Sport',
    'Serie A': 'Sport', 'La Liga': 'Sport', 'Liga e Kampionëve': 'Sport',
    'Superliga': 'Sport', 'Sport vendor': 'Sport',
    
    # Showbiz
    'Showbiz': 'Showbiz', 'Showtime': 'Showbiz', 'Rozë': 'Showbiz',
    'Yjet': 'Showbiz', 'Po Flitet': 'Showbiz', 'Magazina': 'Showbiz',
    
    # Ekonomi
    'Ekonomi': 'Ekonomi', 'Biznes': 'Ekonomi', 'Financa': 'Ekonomi', 'Marketing': 'Ekonomi',
    
    # Kulturë
    'Kulturë': 'Kulturë', 'Muzikë': 'Kulturë',
    
    # Lifestyle
    'Lifestyle': 'Lifestyle',
    
    # Shëndet
    'Shëndet': 'Shëndet', 'Shëndetësi': 'Shëndet', 'Shëndetësi Lajme': 'Shëndet',
    
    # Kuriozitete
    'Kuriozitete': 'Kuriozitete', 'Interesante': 'Kuriozitete', 'Fun': 'Kuriozitete', 'Cool': 'Kuriozitete',
    
    # Politikë
    'Politikë': 'Politikë', 'Politike': 'Politikë',
    
    # Botë
    'Bota': 'Botë', 'Botë': 'Botë', 'Nga Bota': 'Botë',
}

print(f"Total variante te mapuara: {len(KATEGORI_MAPPING)}")
print(f"Kategori finale ({len(set(KATEGORI_MAPPING.values()))}):", sorted(set(KATEGORI_MAPPING.values())))

Total variante te mapuara: 33
Kategori finale (9): ['Botë', 'Ekonomi', 'Kulturë', 'Kuriozitete', 'Lifestyle', 'Politikë', 'Showbiz', 'Shëndet', 'Sport']


## 2. Skanimi i plotë dhe mbledhja e dataset-it të balancuar

(Skanon 5.28M rreshtat në chunks, aplikon mapping-un, mban vetëm artikuj single-label, dhe mbledh deri në një limit të balancuar për kategori.)

In [3]:
LIMIT_PER_KATEGORI = 12000
MAX_PER_PORTAL_PER_KATEGORI = 4500  # asnjë portal s'mund të japë më shumë se 25% e një kategorie
CHUNK_SIZE = 200000

kategorite_finale = sorted(set(KATEGORI_MAPPING.values()))
mbledhur = {kat: [] for kat in kategorite_finale}
portal_counts = {kat: {} for kat in kategorite_finale}  # ndjek sa morëm nga çdo portal, për kategori

def total_mbledhur():
    return sum(sum(len(df_i) for df_i in v) for v in mbledhur.values())

reader = pd.read_csv(r'C:\Users\denis\Downloads\Desktop\diploma-project\data\raw\Kosovo-News-Articles.csv', chunksize=CHUNK_SIZE)

for i, chunk in enumerate(reader):
    chunk = chunk.dropna(subset=['category'])
    chunk['category_clean'] = chunk['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')
    chunk = chunk[chunk['category_clean'] != '']
    chunk['n_labels'] = chunk['category_clean'].str.split(';').apply(len)
    
    single = chunk[chunk['n_labels'] == 1].copy()
    single['kategoria_finale'] = single['category_clean'].map(KATEGORI_MAPPING)
    single = single.dropna(subset=['kategoria_finale'])
    
    for kat in kategorite_finale:
        aktual = sum(len(df_i) for df_i in mbledhur[kat])
        if aktual >= LIMIT_PER_KATEGORI:
            continue
        
        kandidatet = single[single['kategoria_finale'] == kat]
        
        for portal, grupi_portal in kandidatet.groupby('source'):
            aktual_portal = portal_counts[kat].get(portal, 0)
            if aktual_portal >= MAX_PER_PORTAL_PER_KATEGORI:
                continue
            
            aktual = sum(len(df_i) for df_i in mbledhur[kat])
            nevojshem_total = LIMIT_PER_KATEGORI - aktual
            nevojshem_portal = MAX_PER_PORTAL_PER_KATEGORI - aktual_portal
            marr = min(nevojshem_total, nevojshem_portal, len(grupi_portal))
            
            if marr > 0:
                rreshta = grupi_portal.head(marr)
                mbledhur[kat].append(rreshta)
                portal_counts[kat][portal] = aktual_portal + marr
    
    if i % 5 == 0:
        print(f"Chunk {i}, total mbledhur deri tani: {total_mbledhur()}")
    
    if all(sum(len(df_i) for df_i in mbledhur[kat]) >= LIMIT_PER_KATEGORI for kat in kategorite_finale):
        print(f"Mjaftueshëm shembuj u gjetën (chunk {i}), ndalojmë.")
        break

df_final = pd.concat([pd.concat(v) for v in mbledhur.values() if v], ignore_index=True)

print("\n" + "="*40)
print("Shape final:", df_final.shape)
print("\nShpërndarja finale (kategori):")
print(df_final['kategoria_finale'].value_counts())

print("\nShpërndarja kategori × portal (diversiteti):")
print(pd.crosstab(df_final['kategoria_finale'], df_final['source']))

Chunk 0, total mbledhur deri tani: 13500
Chunk 5, total mbledhur deri tani: 47916
Chunk 10, total mbledhur deri tani: 64237
Chunk 15, total mbledhur deri tani: 89479
Chunk 20, total mbledhur deri tani: 104435
Mjaftueshëm shembuj u gjetën (chunk 23), ndalojmë.

Shape final: (108000, 10)

Shpërndarja finale (kategori):
kategoria_finale
Botë           12000
Ekonomi        12000
Kulturë        12000
Kuriozitete    12000
Lifestyle      12000
Politikë       12000
Showbiz        12000
Shëndet        12000
Sport          12000
Name: count, dtype: int64

Shpërndarja kategori × portal (diversiteti):
source            Ballkani  Bota_al  GazetaBlic  GazetaExpress  Indeksonline  \
kategoria_finale                                                               
Botë                     0     4211           0              0             0   
Ekonomi                  0      541           0              0          3756   
Kulturë                  0        0           0              0            34   
Kur

## 3. Ruajtja e dataset-it final

(Ruan `dataset_final.csv` te `data/processed/` — ky është dataset-i që do përdoret për preprocessing, baseline, dhe fine-tuning.)

In [ ]:
df_final.to_csv('../data/processed/dataset_final.csv', index=False)
print(f"U ruajt: data/processed/dataset_final.csv ({df_final.shape[0]} rreshta)")

U ruajt: data/processed/dataset_final.csv (108000 rreshta)


## Instrumentimi i pipeline-it — numrat real për Tabelën 1 (funnel)

Skanon te gjithe 5.283.843 rreshtat (pa ndalim të hershëm) dhe regjistron 
sa mbeten pas secilit hap individual.

In [10]:
import pandas as pd

KATEGORI_MAPPING = {
    'Sport': 'Sport', 'Sporti': 'Sport', 'Premier League': 'Sport',
    'Serie A': 'Sport', 'La Liga': 'Sport', 'Liga e Kampionëve': 'Sport',
    'Superliga': 'Sport', 'Sport vendor': 'Sport',
    'Showbiz': 'Showbiz', 'Showtime': 'Showbiz', 'Rozë': 'Showbiz',
    'Yjet': 'Showbiz', 'Po Flitet': 'Showbiz', 'Magazina': 'Showbiz',
    'Ekonomi': 'Ekonomi', 'Biznes': 'Ekonomi', 'Financa': 'Ekonomi', 'Marketing': 'Ekonomi',
    'Kulturë': 'Kulturë', 'Muzikë': 'Kulturë',
    'Lifestyle': 'Lifestyle',
    'Shëndet': 'Shëndet', 'Shëndetësi': 'Shëndet', 'Shëndetësi Lajme': 'Shëndet',
    'Kuriozitete': 'Kuriozitete', 'Interesante': 'Kuriozitete', 'Fun': 'Kuriozitete', 'Cool': 'Kuriozitete',
    'Politikë': 'Politikë', 'Politike': 'Politikë',
    'Bota': 'Botë', 'Botë': 'Botë', 'Nga Bota': 'Botë',
}

CHUNK_SIZE = 200000
total_original = 0
total_after_ballina = 0
total_single_label = 0
total_after_mapping = 0

reader = pd.read_csv(r'C:\Users\denis\Downloads\Desktop\diploma-project\data\raw\Kosovo-News-Articles.csv', chunksize=CHUNK_SIZE)

for i, chunk in enumerate(reader):
    total_original += len(chunk)

    chunk = chunk.dropna(subset=['category'])
    chunk['category_clean'] = chunk['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')
    after_ballina = chunk[chunk['category_clean'] != ''].copy()
    total_after_ballina += len(after_ballina)

    after_ballina['n_labels'] = after_ballina['category_clean'].str.split(';').apply(len)
    single = after_ballina[after_ballina['n_labels'] == 1].copy()
    total_single_label += len(single)

    single['kategoria_finale'] = single['category_clean'].map(KATEGORI_MAPPING)
    mapped = single.dropna(subset=['kategoria_finale'])
    total_after_mapping += len(mapped)

    if i % 5 == 0:
        print(f"Chunk {i} procesuar... (deri tani: {total_original:,} rreshta lexuar)")

print("\n" + "="*50)
print(f"1. Koleksioni origjinal:              {total_original:,}")
print(f"2. Pas heqjes së etiketës 'Ballina':   {total_after_ballina:,}")
print(f"3. Pas filtrimit single-label:         {total_single_label:,}")
print(f"4. Pas mapimit të kategorive (33→9):   {total_after_mapping:,}")
print(f"5. Pas kufizimit për-portal (final):   108.000")

Chunk 0 procesuar... (deri tani: 200,000 rreshta lexuar)
Chunk 5 procesuar... (deri tani: 1,200,000 rreshta lexuar)
Chunk 10 procesuar... (deri tani: 2,200,000 rreshta lexuar)
Chunk 15 procesuar... (deri tani: 3,200,000 rreshta lexuar)
Chunk 20 procesuar... (deri tani: 4,200,000 rreshta lexuar)
Chunk 25 procesuar... (deri tani: 5,200,000 rreshta lexuar)

1. Koleksioni origjinal:              5,283,840
2. Pas heqjes së etiketës 'Ballina':   5,235,655
3. Pas filtrimit single-label:         2,983,441
4. Pas mapimit të kategorive (33→9):   1,048,368
5. Pas kufizimit për-portal (final):   108.000
